# Module 09 - 실습 1 솔루션: asyncio 비동기 호출

## 학습 목표
- `async`/`await` 키워드를 이해하고 비동기 함수를 작성할 수 있다
- `asyncio.gather()`로 여러 작업을 동시에 실행할 수 있다
- 동기 vs 비동기 실행 시간 차이를 체감할 수 있다

## 참조 자료
- 학습 문서: `docs/part4-production/09-external-system-integration.md` (섹션 1.3, 2.1)

> **Jupyter 주의**: Jupyter 환경에서는 이미 이벤트 루프가 실행 중입니다.
> `asyncio.run()` 대신 셀에서 `await`를 직접 사용하세요.

## 개념 요약

**동기 실행**: 한 작업이 끝날 때까지 다음 작업이 기다림

**비동기 실행**: 여러 작업이 동시에 진행됨

핵심 패턴:
```python
async def my_func():      # 비동기 함수 정의
    await asyncio.sleep(1)  # 비동기 대기
    return "완료"

# 동시 실행
results = await asyncio.gather(my_func(), my_func())
```

In [ ]:
import sys
sys.path.insert(0, '../..')

import asyncio
import time

print("asyncio 준비 완료")

## 실습 1 솔루션: 비동기 함수 작성

In [ ]:
async def fetch_data(name: str, delay: float) -> str:
    """delay초 동안 대기 후 결과 반환 (API 호출 시뮬레이션)."""
    print(f"[{name}] 요청 시작...")
    await asyncio.sleep(delay)  # 비동기 대기
    print(f"[{name}] 응답 수신!")
    return f"{name}: 완료"

In [ ]:
# 검증 셀
result = await fetch_data("TestAPI", 0.1)
assert result == "TestAPI: 완료", f"기대값: 'TestAPI: 완료', 실제값: '{result}'"
print("✅ 실습 1 완료! fetch_data 함수가 올바르게 작동합니다.")

## 실습 2 솔루션: asyncio.gather로 동시 실행

In [ ]:
start = time.time()

# asyncio.gather로 3개 작업 동시 실행
results = await asyncio.gather(
    fetch_data("Jira", 2.0),
    fetch_data("GitLab", 1.5),
    fetch_data("Slack", 1.0),
)

elapsed = time.time() - start
print(f"\n결과: {results}")
print(f"소요 시간: {elapsed:.1f}초")

In [ ]:
# 검증 셀
assert results is not None, "results가 None입니다."
assert len(results) == 3, f"결과가 3개여야 합니다. 현재: {len(results)}개"
assert "Jira: 완료" in results, f"Jira 결과가 없습니다: {results}"
assert "GitLab: 완료" in results, f"GitLab 결과가 없습니다: {results}"
assert "Slack: 완료" in results, f"Slack 결과가 없습니다: {results}"
assert elapsed < 3.0, f"소요 시간이 너무 깁니다 ({elapsed:.1f}초)."
print(f"✅ 실습 2 완료! {elapsed:.1f}초 만에 3개 API 호출 완료.")

## 실습 3 솔루션: 동기 vs 비동기 성능 비교

In [ ]:
start_sync = time.time()

# 순차 실행 (gather 없이 await를 하나씩 호출)
result1 = await fetch_data("Jira", 2.0)
result2 = await fetch_data("GitLab", 1.5)
result3 = await fetch_data("Slack", 1.0)
sync_results = [result1, result2, result3]

elapsed_sync = time.time() - start_sync
print(f"순차 실행 결과: {sync_results}")
print(f"순차 실행 소요 시간: {elapsed_sync:.1f}초")

In [ ]:
# 검증 셀
assert sync_results is not None, "sync_results를 구현하세요."
assert len(sync_results) == 3, f"순차 실행 결과가 3개여야 합니다."
assert elapsed_sync >= 4.0, f"순차 실행은 최소 4.5초가 필요합니다. 현재: {elapsed_sync:.1f}초"
print(f"\n성능 비교:")
print(f"  순차 실행: {elapsed_sync:.1f}초")
print(f"  동시 실행: {elapsed:.1f}초")
print(f"  속도 향상: {elapsed_sync/elapsed:.1f}배")
print("✅ 실습 3 완료! 동기 vs 비동기 성능 차이를 확인했습니다.")

## 정리

이번 실습에서 배운 내용:
1. `async def`로 비동기 함수를 정의하고 `await`로 호출한다
2. `asyncio.gather()`로 여러 비동기 작업을 동시에 실행한다
3. 비동기 실행은 총 시간 = max(각 작업 시간)이므로 훨씬 빠르다

## 다음 모듈
- **실습 2**: `02_rate_limiter.ipynb` - TokenBucket으로 API 호출 속도 제한